In [1]:
!pip install -U langchain
!pip install -U langchain-cohere
!pip install -U langchain-community
!pip install -U langchain-text-splitters
!pip install -U langchain-chroma
!pip install -U pypdf
!pip install -U chromadb
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 12.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.3.13
    Uninstalling langgraph-sdk-0.3.13:
      Successfully uninstalled langgraph-sdk-0.3.13
  Attempting uninstall: langgraph-checkpoint
    Found existing installation: langgraph-checkpoint 4.0.2
    Uninstalling langgraph-checkpoint-4.0.2:
    

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
key = user_secrets.get_secret("coherekey")

----

## **Part 1 — Test the RAG with Your Own CV**

In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/kaggle/input/datasets/islamohamed10/my-cv-v2/CV.pdf")

documents = loader.load()

print("Number of pages:", len(documents))

print(documents[0].page_content[:500])

/tmp/ipykernel_58/3778375639.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Number of pages: 2
Islam Mohamed
♂¶ap-¶arker-altCairo, Egypt✉10islammohamed01@gmail.com♂phone-alt+20 114 468 2583/linkedin-inIslam Mohamed
/githubislam0114
Summary
Computer and Communication Engineering student at Shoubra Engineering with a strong pas-
sion for Artificial Intelligence and Machine Learning. Actively building skills in machine learning,
deep learning, data preprocessing, and data analysis through intensive training programs and cer-
tifications from IBM, Microsoft, NTI, ITI, DEPI, and DataCamp. Know


In [4]:
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=key
)


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

In [6]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="pdf_rag"
)

In [7]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)

In [8]:
query = "What is this document about?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\n--- Document {i+1} ---")

    print(doc.page_content)

    print("Metadata:", doc.metadata)


--- Document 1 ---
Islam Mohamed
♂¶ap-¶arker-altCairo, Egypt✉10islammohamed01@gmail.com♂phone-alt+20 114 468 2583/linkedin-inIslam Mohamed
/githubislam0114
Summary
Computer and Communication Engineering student at Shoubra Engineering with a strong pas-
sion for Artificial Intelligence and Machine Learning. Actively building skills in machine learning,
deep learning, data preprocessing, and data analysis through intensive training programs and cer-
tifications from IBM, Microsoft, NTI, ITI, DEPI, and DataCamp. Known for being organized,
disciplined with time, and a critical thinker who enjoys learning and exploring new AI concepts.
Currently seeking internship or junior roles to gain practical experience, apply theoretical knowl-
edge, and grow into a distinguished professional in the AI field.
Education
Metadata: {'page': 0, 'creationdate': '2026-06-12T20:06:07+00:00', 'title': "Islam Mohamed's CV", 'keywords': '', 'page_label': '1', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159

In [9]:
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    cohere_api_key=key
)


In [10]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the context below.

If the answer cannot be found in the context,
say "I don't know."

Context:
{context}## top 1

Question:
{question}
""")

In [11]:
from langchain_core.runnables import RunnablePassthrough

In [12]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [13]:
question = "•	What are my main technical skills?"

response = rag_chain.invoke(question)

print(response.content)

Your main technical skills, as outlined in the context, include:

- **Programming**: Python, Pandas, NumPy, Matplotlib, Seaborn, SciKit-learn, TensorFlow, Keras, PyTorch  
- **Machine Learning**: Supervised and Unsupervised Learning Algorithms, Model Training and Evaluation, Feature Selection, Hyperparameter Tuning  
- **Deep Learning**: Neural Networks, CNN, RNN, Transformers  
- **Data Science**: Data Preprocessing, EDA, Statistical Analysis, Data Visualization  
- **Deployment Tools**: Git, GitHub, Streamlit


In [14]:
question = "•	What machine learning experience do I have?"

response = rag_chain.invoke(question)

print(response.content)

Based on the context provided, your machine learning experience includes:

1. **Supervised and Unsupervised Learning**: Developed and fine-tuned models using algorithms like Random Forest, with experience in feature selection, hyperparameter tuning, and model evaluation.  
2. **Deep Learning**: Explored architectures such as CNN (Convolutional Neural Networks) and RNN (Recurrent Neural Networks), as well as Transformers.  
3. **Model Deployment**: Built deployment-ready solutions using Streamlit and Power BI for real-time insights.  
4. **Projects**:  
   - **Bank Customer Churn**: Developed a Random Forest model to predict customer churn using demographic, financial, and behavioral data.  
   - **Moshrif – Smart Attendance & University ERP System**: Built the core AI vision engine for a 4-layer microservices ERP system.  
   - **Bilingual Semantic Chatbot**: Created a RAG pipeline using Sentence-BERT and Google Gemini API.  
5. **Tools and Libraries**: Utilized Python, Scikit-learn, T

In [15]:
question = "•	What projects are mentioned in my CV?"

response = rag_chain.invoke(question)

print(response.content)

The projects mentioned in your CV are:

1. **Moshrif – Smart Attendance & University ERP System**  
   - Built the core AI vision engine for a 4-layer microservices ERP system to automate university attendance operations.  
   - Engineered a real-time face recognition pipeline with EAR liveness detection to eliminate proxy attendance and spoofing.  
   - Integrated the AI model with FastAPI for instant WebSocket data synchronization and automated email alerts.  

2. **Bilingual Semantic Chatbot**  
   - Built a RAG pipeline using paraphrase-multilingual-MiniLM-L12-v2 and Google Gemini API.  
   - Integrated the AI layer asynchronously with FastAPI and MySQL for real-time dashboard updates.  

3. **Bank Customer Churn Project**  
   - Developed and deployed a Random Forest model to predict bank customer churn using demographic, financial, and behavioral data.  

4. **BiblioTech: AI-Powered Smart Library Management System**  
   - Developed a hybrid recommendation engine trained on book 

In [16]:
question = "•	What programming languages do I know?"

response = rag_chain.invoke(question)

print(response.content)

Based on the context, you know the following programming languages and tools:

- **Python**  
- **Pandas**  
- **NumPy**  
- **Matplotlib**  
- **Seaborn**  
- **SciKit-learn**  
- **TensorFlow**  
- **Keras**  
- **PyTorch**  
- **FastAPI**  
- **React.js**  
- **MySQL**  
- **Google Gemini API**  
- **Sentence-BERT**  
- **JWT**  
- **Bcrypt**  
- **Recharts**  

Additionally, you have experience with deployment tools like **Git**, **GitHub**, and **Streamlit**, as well as data visualization tools like **Power BI**.


In [17]:
question = "•	What training or internships have I completed?"

response = rag_chain.invoke(question)

print(response.content)

Based on the context provided, you have completed the following training and internships:

1. **Data Science and Machine Learning Trainee, DEPI** (Dec 2025 – Jul 2026)  
2. **AI Summer Training – ITI** (Aug 2025)  
3. **Machine Learning – NTI** (Aug 2025)  
4. **AI & Machine Learning – Sprints x Microsoft** (Oct 2025)  
5. **IBM AI Engineer Certificate – Coursera** (Jan 2026)  
6. **AI & Data Science – DEPI** (May 2025)  

Additionally, you are currently seeking internship or junior roles to gain further practical experience.


----

## **Part 2 — Ask Questions About a Book**

In [18]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
key = user_secrets.get_secret("coherekey")

In [19]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/kaggle/input/datasets/islamohamed10/python/python crash course.pdf")

documents = loader.load()

print("Number of pages:", len(documents))

print(documents[0].page_content[:500])

Number of pages: 562
A HANDS-ON , PROJECT-BASED
INTRODUCTION TO PROGRAMMING
ERIC MATTHES
P Y THON
C R ASH COURSE
P Y THON
C R ASH COURSE
SHELVE IN:
PROGRAMMING LANGUAGES/
PYTHON
$39.95 ($45.95 CDN)
FAST!
LEARN PYTHON—
FAST!
LEARN PYTHON—
PYTHON CRASH COURSEPYTHON CRASH COURSEMATTHES
COVERS PYTHON 2 AND 3
Python Crash Course is a fast-paced, thorough intro-
duction to programming with Python that will have you 
writing programs, solving problems, and making things 
that work in no time. 
In the first half of the book


In [20]:
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=key
)


In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400
)

chunks = text_splitter.split_documents(documents)

In [22]:
from langchain_chroma import Chroma
import time

# To avoid the 429 Too Many Requests error with Trial keys,
# we decrease batch size and increase sleep time to stay under token limits.
batch_size = 30
vectorstore = Chroma(collection_name="pdf_rag", embedding_function=embeddings)

for i in range(0, len(chunks), batch_size):
    batch = chunks[i : i + batch_size]
    vectorstore.add_documents(batch)
    print(f"Processed {i + len(batch)} / {len(chunks)} chunks...")
    if i + batch_size < len(chunks):
        # Trial keys often require a significant pause to reset token-per-minute counters
        time.sleep(7)

Processed 30 / 806 chunks...
Processed 60 / 806 chunks...
Processed 90 / 806 chunks...
Processed 120 / 806 chunks...
Processed 150 / 806 chunks...
Processed 180 / 806 chunks...
Processed 210 / 806 chunks...
Processed 240 / 806 chunks...
Processed 270 / 806 chunks...
Processed 300 / 806 chunks...
Processed 330 / 806 chunks...
Processed 360 / 806 chunks...
Processed 390 / 806 chunks...
Processed 420 / 806 chunks...
Processed 450 / 806 chunks...
Processed 480 / 806 chunks...
Processed 510 / 806 chunks...
Processed 540 / 806 chunks...
Processed 570 / 806 chunks...
Processed 600 / 806 chunks...
Processed 630 / 806 chunks...
Processed 660 / 806 chunks...
Processed 690 / 806 chunks...
Processed 720 / 806 chunks...
Processed 750 / 806 chunks...
Processed 780 / 806 chunks...
Processed 806 / 806 chunks...


In [23]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)

In [24]:
query = "What is this document about?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\n--- Document {i+1} ---")

    print(doc.page_content)

    print("Metadata:", doc.metadata)


--- Document 1 ---
xxvi   Contents in Detail
IDLE  .  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 496
Installing IDLE on Linux . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 496
Installing IDLE on OS X  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 496
Installing IDLE on Windows  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 497
Customizing IDLE Settings  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 497
Emacs and vim  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 497
C 
gettIng helP 499
First Steps  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 499
Try It Again  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

In [25]:
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    cohere_api_key=key
)


In [26]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the context below.

If the answer cannot be found in the context,
say "I don't know."

Context:
{context}## top 1

Question:
{question}
""")

In [27]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [28]:
question = "what is the list"

response = rag_chain.invoke(question)

print(response.content)

A list is a collection of items in a particular order. You can make a list that includes the letters of the alphabet, the digits from 0–9, or the names of all the people in your family. You can put anything you want into a list. Lists allow you to store sets of information in one place, whether you have just a few items or millions of items. They are one of Python’s most powerful features readily accessible to new programmers, tying together many important concepts in programming.


In [29]:
question = "what is for loop?"

response = rag_chain.invoke(question)

print(response.content)

A for loop in Python is a programming construct that allows you to iterate over a sequence (such as a list, tuple, dictionary, set, or string) and execute a block of code for each item in the sequence. It automates repetitive tasks by letting Python manage the iteration internally, so you don't have to manually retrieve each item from the list or change your code when the list's length changes. 

In the context provided, a for loop is used to print each name in a list of magicians. For example:

```python
magicians = ['alice', 'david', 'carolina']
for magician in magicians:
    print(magician)
```

Here, the for loop iterates over each name in the `magicians` list, assigns it to the variable `magician`, and then prints the name. This process repeats for every item in the list, making it efficient and flexible for working with lists of any length.


In [30]:
question = "who is mohamed salah?"

response = rag_chain.invoke(question)

print(response.content)

I don't know. The context provided does not contain any information about Mohamed Salah. It appears to be a resume or profile for a person named Islam Mohamed, who is a Computer and Communication Engineering student with a focus on Artificial Intelligence and Machine Learning. The rest of the context includes Python code examples and explanations related to programming concepts such as lists, slicing, and loops.


In [31]:
question = "what are the data types in python?"

response = rag_chain.invoke(question)

print(response.content)

Based on the provided context, the data types in Python mentioned are:

1. **Strings**: A series of characters enclosed in single or double quotes.  
2. **Integers**: Whole numbers without decimal points.  
3. **Floats**: Numbers with decimal points.  

These are the primary data types discussed in the context.


In [32]:
question = "how to define a function?"

response = rag_chain.invoke(question)

print(response.content)

To define a function in Python, you use the `def` keyword followed by the function name and parentheses `()`. Inside the parentheses, you can include parameters if the function requires input. The function definition ends with a colon `:`. The body of the function, which contains the code to be executed, is indented below the definition line. Here’s the basic structure:

```python
def function_name(parameters):
    """Docstring: Describe what the function does."""
    # Function body: Code to be executed
    pass
```

For example, from the context:

```python
def greet_user():
    """Display a simple greeting."""
    print("Hello!")
```

In this example:
- `def` is the keyword to define a function.
- `greet_user` is the name of the function.
- The parentheses `()` are empty because this function doesn't require any parameters.
- The colon `:` ends the function definition.
- The indented line `print("Hello!")` is the body of the function.


In [33]:
question = "what is object oriented programming?"

response = rag_chain.invoke(question)

print(response.content)

Object-oriented programming (OOP) is one of the most effective approaches to writing software. In OOP, you write classes that represent real-world things and situations, and you create objects based on these classes. When you write a class, you define the general behavior that a whole category of objects can have. When you create individual objects from the class, each object is automatically equipped with the general behavior, and you can then give each object unique traits. OOP allows you to model real-world situations effectively and helps you understand the world as a programmer, thinking logically to write programs that address almost any problem.


### Chunking Experiment
Testing a different chunk size (chunk_size=800, chunk_overlap=100) and observing the retrieved context.

In [34]:
text_splitter_exp = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks_exp = text_splitter_exp.split_documents(documents)

# Create a temporary vectorstore for the experiment
vectorstore_exp = Chroma.from_documents(chunks_exp[:50], embeddings, collection_name="pdf_rag_exp")

retriever_exp = vectorstore_exp.as_retriever(search_kwargs={"k": 5})

query_exp = "what is the list"
retrieved_docs_exp = retriever_exp.invoke(query_exp)

print("Retrieved chunks for chunk_size=800:")
for i, doc in enumerate(retrieved_docs_exp):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content)

Retrieved chunks for chunk_size=800:

--- Document 1 ---
Exercise 2-11: Zen of Python ................................... 36
Summary  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 36
3 
IntroduCIng lIsts 37
What Is a List? . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 37
Accessing Elements in a List  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 38
Index Positions Start at 0, Not 1  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 39
Using Individual Values from a List  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 39
Exercise 3-1: Names ......................................... 40

--- Document 2 ---
Exercise 3-1: Names ......................................... 40
Exercise 3-2: Greetings ....................................... 40
Exercise 3-3: Your Own List ..................

----

## **Part 3 — Multi-PDF RAG**

In [35]:
from langchain_community.document_loaders import PyPDFLoader
import os

# Define the list of PDF files you want to load
pdf_files = ["/kaggle/input/datasets/islamohamed10/data-structure/Part 2 Linked List.pdf", "/kaggle/input/datasets/islamohamed10/data-structure/Part 3  Stack.pdf","/kaggle/input/datasets/islamohamed10/data-structure/Part 4 Queue.pdf","/kaggle/input/datasets/islamohamed10/data-structure/Part 5 Recursion.pdf"]

all_documents = []

for pdf_path in pdf_files:
    if os.path.exists(pdf_path):
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        all_documents.extend(docs)
        print(f"Loaded {pdf_path}: {len(docs)} pages")
    else:
        print(f"Warning: File not found at {pdf_path}")

print(f"\nTotal documents loaded: {len(all_documents)}")

Loaded /kaggle/input/datasets/islamohamed10/data-structure/Part 2 Linked List.pdf: 29 pages
Loaded /kaggle/input/datasets/islamohamed10/data-structure/Part 3  Stack.pdf: 18 pages
Loaded /kaggle/input/datasets/islamohamed10/data-structure/Part 4 Queue.pdf: 27 pages
Loaded /kaggle/input/datasets/islamohamed10/data-structure/Part 5 Recursion.pdf: 14 pages

Total documents loaded: 88


In [36]:
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=key
)


In [37]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Split all loaded documents into chunks
chunks = text_splitter.split_documents(all_documents)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 113


In [38]:
from langchain_chroma import Chroma
import time

# To avoid the 429 Too Many Requests error with Trial keys,
# we decrease batch size and increase sleep time to stay under token limits.
batch_size = 30
vectorstore = Chroma(collection_name="pdf_rag", embedding_function=embeddings)

for i in range(0, len(chunks), batch_size):
    batch = chunks[i : i + batch_size]
    vectorstore.add_documents(batch)
    print(f"Processed {i + len(batch)} / {len(chunks)} chunks...")
    if i + batch_size < len(chunks):
        # Trial keys often require a significant pause to reset token-per-minute counters
        time.sleep(3)

Processed 30 / 113 chunks...
Processed 60 / 113 chunks...
Processed 90 / 113 chunks...
Processed 113 / 113 chunks...


In [39]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)

In [40]:
query = "What is this document about?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\n--- Document {i+1} ---")

    print(doc.page_content)

    print("Metadata:", doc.metadata)


--- Document 1 ---
Data Structures and Algorithms 
Part 4 : Queue
Data Structures and Algorithms    Part 41
Metadata: {'source': '/kaggle/input/datasets/islamohamed10/data-structure/Part 4 Queue.pdf', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2026-03-05T09:46:18+02:00', 'page_label': '1', 'moddate': '2026-03-05T09:46:18+02:00', 'total_pages': 27, 'title': 'Data Structures and Algorithms    Lecture 6 : Queue', 'author': 'pc', 'page': 0}

--- Document 2 ---
Data Structures and Algorithms
Part 3: Stack 
Data structures and Algorithms   Part 31
Metadata: {'moddate': '2026-02-24T21:06:41+02:00', 'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'source': '/kaggle/input/datasets/islamohamed10/data-structure/Part 3  Stack.pdf', 'creationdate': '2026-02-24T21:06:41+02:00', 'author': 'pc', 'page_label': '1', 'page': 0, 'title': 'Data Structures and Algorithms', 'creator': 'Microsoft® PowerPoint® for Mic

In [41]:
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    cohere_api_key=key
)


In [42]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the context below.

If the answer cannot be found in the context,
say "I don't know."

Context:
{context}## top 1

Question:
{question}
""")

In [43]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [44]:
question = "what is the linkedlist?"

response = rag_chain.invoke(question)

print(response.content)

Based on the provided context, a **LinkedList** is a data structure composed of a sequence of elements called **links** or **nodes**. Each link contains data and a reference (or pointer) to the next link in the sequence. The **LinkList** class, for example, contains a reference to the first link (`first`) in the list. 

Here are the key characteristics of a LinkedList from the context:

1. **Link Class**: Each link in the list is an instance of the `Link` class, which contains:
   - Data items (e.g., `iData`, `dData`, or `dData` depending on the implementation).
   - A reference to the next link (`next`).

2. **LinkList Class**: The `LinkList` class manages the list and contains:
   - A reference to the first link (`first`).
   - Methods to check if the list is empty (`isEmpty()`).

3. **FirstLastList Class**: An extension of the basic LinkedList, which also keeps a reference to the last link (`last`) for efficient insertion and deletion at the end of the list.

In summary, a **LinkedL

In [45]:
question = "what is the stack?"

response = rag_chain.invoke(question)

print(response.content)

A stack is a LIFO (Last In, First Out) data structure that allows access to only one data item: the last item inserted. If you remove this item, you can access the next-to-last item inserted, and so on.


In [46]:
question = "who is mohamed salah?"

response = rag_chain.invoke(question)

print(response.content)

I don't know. The context provided does not contain any information about Mohamed Salah. It appears to be a resume or profile for a person named Islam Mohamed, who is a Computer and Communication Engineering student with a focus on Artificial Intelligence and Machine Learning. The rest of the context seems to be related to Python programming, specifically working with lists and slices.


## **Bonus**

In [17]:
import os
import time
import re

print("Killing old processes...")
os.system("pkill -f streamlit")
os.system("pkill -f cloudflared")
time.sleep(2)

print("Starting new Streamlit app...")
os.system("python -m streamlit run /kaggle/input/datasets/islamohamed10/streamlit-v10/app.py --server.enableCORS false --server.enableXsrfProtection false &")

print("Downloading and Starting Cloudflare Tunnel...")
os.system("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
os.system("chmod +x cloudflared-linux-amd64")

os.system("rm -f tunnel.log")
os.system("nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8501 > tunnel.log 2>&1 &")

print("Waiting for the tunnel to connect (5-10 seconds)...")
time.sleep(8)

with open("tunnel.log", "r") as f:
    logs = f.read()
    url = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", logs)
    if url:
        print("your url", url.group())
    else:
        print("!grep -o 'https://.*\.trycloudflare\.com' tunnel.log")


<>:29: SyntaxWarning: invalid escape sequence '\.'
<>:29: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_58/1449112660.py:29: SyntaxWarning: invalid escape sequence '\.'
  print("!grep -o 'https://.*\.trycloudflare\.com' tunnel.log")


Killing old processes...
  Stopping...
Starting new Streamlit app...
Waiting for the tunnel to connect (5-10 seconds)...




2026-08-11 17:36:45.609 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.187.146.115:8501

your url https://shortcuts-hunter-ware-monte.trycloudflare.com


/kaggle/input/datasets/islamohamed10/streamlit-v10/app.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
